<a href="https://colab.research.google.com/github/sjayavelu73/langgraph/blob/lang1/RAG_langgraph_ingestion_of_two_pdfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#!pip install langchain_core
#!pip install langchain_community
#!pip install langchain_openai
#!pip install langchain_chroma
#!pip install langgraph
#!pip install PyMUPDF


# Import Packages
from langchain.chains import create_retrieval_chain
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langgraph.graph import START,END,StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage,AIMessage,HumanMessage,BaseMessage, ToolMessage
from typing import TypedDict,Union,Sequence,Annotated
from google.colab import drive
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import os
import fitz
from langchain_core.tools import tool
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Load OpenAI API Key
# Make sure to add your OpenAI API key to Colab's user data secrets with the name 'OPENAI_API_KEY'
os.environ['OPENAI_API_KEY']=userdata.get('ai_agents_openai')


# Load and process the pdfs
drive.mount('/content/drive')
file_paths=['/content/drive/MyDrive/Malaria.pdf', '/content/drive/MyDrive/RAG.pdf']
pdf_splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=300)

all_docs=[]
for file_path in file_paths:
  pdfdoc=fitz.open(file_path)
  all_page_text=[]
  for page_num in pdfdoc:
    page_text=page_num.get_text()
    all_page_text.append(page_text)

  for i, page_text in enumerate(all_page_text):
    chunks = pdf_splitter.split_text(page_text)
    docs = [Document(page_content=chunk, metadata={"page": i+1, "source": os.path.basename(file_path)}) for chunk in chunks]
    all_docs.extend(docs)


embeddings=OpenAIEmbeddings()
db=Chroma.from_documents(all_docs,embeddings)
retriever = db.as_retriever()

llm_with_no_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class RagState(TypedDict):
  messages: Annotated[Sequence[BaseMessage],add_messages]


@tool
def retrieval_from_pdfs(query: str) -> str:
  """
  Searches the loaded documents for information related to the query.
  """
  # Define prompt template
  prompt = ChatPromptTemplate.from_messages([
      ("system", "Answer the question based on the provided context: {context}"),
      ("human", "{query}"),
  ])

  # Get retrieved documents
  retrieved_docs = retriever.invoke(query)

  # Create a chain using LCEL
  document_processing_chain = (
      prompt
      | llm_with_no_tools
  )

  # Invoke the document processing chain with retrieved docs and query
  response = document_processing_chain.invoke({"context": retrieved_docs, "query": query})

  # Include source information in the response
  source_info = []
  for doc in retrieved_docs:
      source = doc.metadata.get("source", "Unknown Source")
      page = doc.metadata.get("page", "Unknown Page")
      source_info.append(f"{source} (Page {page})")

  # Return the response content and source information
  return f"{response.content}\n\nSources: {', '.join(source_info)}"


def model_invoke(state: RagState) -> RagState:
  """
  Invokes the language model to generate a response or decide on a tool.
  """
  response = llm_with_tools.invoke(state["messages"])
  return {"messages": [response]}

# Define the tool node
tools=[retrieval_from_pdfs] # Define tools list before binding
llm_with_tools = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

tool_node = ToolNode(tools)

# Build the graph
workflow = StateGraph(RagState)

workflow.add_node("model", model_invoke)
workflow.add_node("tool", tool_node)

workflow.add_edge(START, "model")

# Define the conditional edge
def should_continue(state: RagState) -> str:
    messages = state['messages']
    last_message = messages[-1]
    # If there is no tool call, then we finish
    if not last_message.tool_calls:
        return "end"
    # Otherwise if there is a tool call, we go to the tool node
    return "continue"

workflow.add_conditional_edges(
    "model",
    should_continue,
    {
        "continue": "tool",
        "end": END
    }
)

workflow.add_edge("tool", "model")

# Compile the graph
app = workflow.compile()

# Run the graph with sample queries
inputs_list = [
    {"messages": [HumanMessage(content="What is the recommended treatment for malaria in pregnant women?")]},
    {"messages": [HumanMessage(content="What is the treatment for chloroquine resistant malaria?")]},
    {"messages": [HumanMessage(content="what are the complications of  babies born to mothers with malaria ")]},
    {"messages": [HumanMessage(content="What is a good chunk size in RAG?")]} # Added a query related to the new document
]

for inputs in inputs_list:
    for output in app.stream(inputs):
        for key, value in output.items():
            print(f"Output from node '{key}':")
            print("----")
            # Check if the output is from the model and contains a tool call result
            if key == 'model' and value['messages'] and isinstance(value['messages'][0], AIMessage) and value['messages'][0].tool_calls:
                 print("Model decided to use a tool.")
                 # The actual tool output will be in the next node ('tool')
            elif key == 'tool' and value['messages'] and isinstance(value['messages'][0], ToolMessage):
                print(f"Tool Output:")
                print(value['messages'][0].content)
            elif key == 'model' and value['messages'] and isinstance(value['messages'][0], AIMessage) and not value['messages'][0].tool_calls:
                 print("Model answered based on its internal knowledge:")
                 print(value['messages'][0].content)
            else:
                print(value)
        print("\n---\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output from node 'model':
----
Model decided to use a tool.

---

Output from node 'tool':
----
Tool Output:
The recommended treatment for severe malaria in pregnant women includes the following:

1. **Parenteral antimalarial drugs** should be administered in full doses without delay.
2. **Parenteral artesunate** is the treatment of choice for all trimesters of pregnancy.
3. If artesunate is unavailable, **intramuscular artemether** should be given.
4. If both artesunate and artemether are unavailable, **parenteral quinine** should be started immediately until artesunate can be obtained.

It is crucial that treatment is not delayed, as pregnant women are at a higher risk of severe malaria and associated complications.

Sources: Malaria.pdf (Page 95), Malaria.pdf (Page 95), Malaria.pdf (Page 95), Malaria.pdf (Page 95)

---

Output from node 'model':
----
Model